[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/su-ntu-ctp/6m-data-3.10-Transformers-Attention-GenAI/blob/main/notebooks/assignment.ipynb)

**Where to run this notebook**

- **Locally (VS Code + Jupyter)**: just open the notebook and pick the `dsai-m3` kernel. The repo's `.env` file handles thread caps automatically.
- **Colab (recommended if you don't have a GPU)**: click the badge above, then **Runtime → Change runtime type → T4 GPU**, then run the setup cell below. It clones the repo, installs missing deps, and `cd`s into the right working directory.


In [7]:
# === Colab-compat setup (no-op when running locally) ===
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/su-ntu-ctp/6m-data-3.10-Transformers-Attention-GenAI.git"
    REPO_DIR = "/content/6m-data-3.10-Transformers-Attention-GenAI"
    LESSON_DIR = "notebooks"

    if not os.path.exists(REPO_DIR):
        print(f"Cloning repo into {REPO_DIR} ...")
        os.system(f"git clone -q {REPO_URL} {REPO_DIR}")

    os.chdir(f"{REPO_DIR}/{LESSON_DIR}")
    print(f"Working directory: {os.getcwd()}")

    # Colab has torch + torchvision pre-installed. Install the rest.
    os.system("pip install -q sentence-transformers transformers")
    print("Colab setup done.")

# Threading caps — picked up by .env locally, set explicitly here as a fallback.
# Harmless if already set. (Loop form prevents Jupyter from auto-displaying the return value.)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")   # silence a harmless warning
# (Old OMP/MKL one-thread caps removed — they slowed CPU-only machines.)


# L10 · Assignment — Ship a RAG shopping assistant

> ⏱️ **~60–90 min** self-study (one track only) · model runtime is ~5–10 min; the rest is your thinking and grading

> *Marcus has approved the project. Your task: build a working RAG shopping assistant that can answer customer questions about the NorthStar catalogue. Then evaluate it on a small benchmark and identify two specific improvement areas.*

In this assignment you will:

**Part A.** Implement a `RAGSystem` from scratch (you can adapt NB 04's class). The system should:
- Retrieve top-K relevant products by semantic similarity
- Pass them to an LLM with a constrained system prompt
- Return both the answer AND the retrieved products (audit trail)

**Part B.** Run the system on a 5-query benchmark and grade the output.

**Part C.** Identify two specific weaknesses of your RAG system and describe how you would fix them.

Total runtime: ~5-10 minutes.

---

## Setup

In [8]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # allow duplicate OpenMP runtime (macOS libomp conflict)
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM

# (Removed the old one-CPU-thread cap — it made CPU-only machines much slower.)
torch.manual_seed(0)

df = pd.read_csv('data/northstar_catalogue.csv')
print(f"Catalogue: {len(df)} products")

retriever = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = AutoTokenizer.from_pretrained('HuggingFaceTB/SmolLM2-360M-Instruct')
llm = AutoModelForCausalLM.from_pretrained('HuggingFaceTB/SmolLM2-360M-Instruct')
print('Models loaded.')

Catalogue: 76 products


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Models loaded.


---

## 📚 Choose your track

This assignment has **two tracks**. Pick **one** based on your background — you don't need to do both.

| Track | Who it's for | What you'll do |
|---|---|---|
| **🟢 Foundational Track** | Learners new to ML / programming | Part A — implement your RAGSystem class |
| **🔵 Advanced Track** | Learners with prior ML background | Part B — benchmark on 5 queries + Part C identify two improvements |

If you're unsure, start with the **Foundational Track**. If it feels easy, skip ahead to the **Advanced Track** — both tracks cover the same lesson outcomes; only the scaffolding differs.

---


---

# 🟢 Foundational Track

> *No prior ML background needed. The cells below are scaffolded — read the worked example, then fill in the blanks. Hints are included.*

---


---

## Part A — Implement your RAGSystem

The class should have at least: `__init__`, `retrieve`, `ask`. Adapt NB 04's class or write your own.
<details>
<summary>💡 Stuck on where to start? Click for a step-by-step plan</summary>

Build it in the same three moves as NB 04 — each maps to one method:

1. **`__init__`** — embed the catalogue **once** up front. Combine name + description + price into one string per product, then `retriever.encode(...)`. (Why: embedding 76 products every query would be wasteful.)
2. **`retrieve(query, top_k)`** — embed the query, `cosine_similarity` against the stored embeddings, take the top-K rows. (This is L09's semantic search, unchanged.)
3. **`ask(query)`** — format the retrieved rows as a bullet list, put them in the **user turn** with a "recommend ONLY from these" instruction, wrap with `apply_chat_template`, and `generate`.

If any step feels foggy, re-run the matching section of NB 04 first — this class is that notebook, condensed.
</details>

In [9]:
class RAGSystem:
    """Shopping assistant: retrieve relevant products, then generate a grounded answer.

    Design notes (mirrors NB 04):
    - Catalogue goes in the USER turn, not the system prompt. A 360M-parameter
      model drifts on long structured-data system prompts.
    - repetition_penalty=1.15 prevents degenerate loops where the model
      repeats the same fabricated product line.
    """

    def __init__(self, df, retriever, llm, tokenizer):
        self.df = df.reset_index(drop=True).copy()
        self.retriever = retriever
        self.llm = llm
        self.tok = tokenizer
        docs = (self.df['name'] + ' — ' + self.df['description']
                + ' (£' + self.df['price_gbp'].astype(str) + ')').tolist()
        self.embeddings = self.retriever.encode(docs, show_progress_bar=False)

    def retrieve(self, query, top_k=5):
        q_emb = self.retriever.encode([query])
        sims = cosine_similarity(q_emb, self.embeddings)[0]
        idx = np.argsort(-sims)[:top_k]
        return self.df.iloc[idx].copy().assign(similarity=sims[idx]).reset_index(drop=True)

    def ask(self, query, top_k=5, max_new_tokens=180):
        retrieved = self.retrieve(query, top_k=top_k)
        lines = [f"- [{r['product_id']}] {r['name']} ({r['category']}, £{r['price_gbp']}): {r['description']}"
                 for _, r in retrieved.iterrows()]
        catalogue_block = '\n'.join(lines)

        sys_msg = (
            "You are a helpful retail shopping assistant for NorthStar. "
            "You ONLY recommend products that the user has explicitly listed below. "
            "Do not invent products. Be concise — one or two sentences plus the product names."
        )
        user_msg = (
            f"Customer question: \"{query}\"\n\n"
            f"Available products from our catalogue:\n{catalogue_block}\n\n"
            "Which one or two would you recommend? Reply with product names and a brief reason."
        )
        messages = [
            {'role': 'system', 'content': sys_msg},
            {'role': 'user',   'content': user_msg},
        ]
        prompt = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        input_ids = self.tok(prompt, return_tensors='pt')['input_ids']
        out = self.llm.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=self.tok.eos_token_id,
                                repetition_penalty=1.15)
        answer = self.tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        return {'query': query, 'answer': answer, 'retrieved': retrieved}

rag = RAGSystem(df, retriever, llm, tokenizer)
print('RAGSystem ready.')

RAGSystem ready.


---

# 🔵 Advanced Track

> *For learners with prior ML background. Minimal scaffolding — you decide the approach. You're welcome to peek at the Foundational Track above for reference.*

---


---

## Part B — Benchmark the system

*Why this matters: a RAG demo that answers one query well tells you almost nothing — the most expensive GenAI failures ship on the strength of a single good demo. A small benchmark is the cheapest insurance you can buy.*

Run 5 plausible customer queries through the system. Evaluate each on three criteria.

In [10]:
BENCHMARK = [
    "I need a warm winter coat for under £200",
    "Something to wear to a summer beach holiday",
    "What's a smart but comfortable shirt for the office?",
    "Recommend a gym outfit for cold-weather running",
    "I'm looking for a wedding-guest dress",
]

results = []
for q in BENCHMARK:
    t0 = time.time()
    r = rag.ask(q)
    elapsed = time.time() - t0
    results.append({'query': q, 'answer': r['answer'], 'retrieved': r['retrieved'], 'elapsed': elapsed})

# Pretty-print each result
for i, r in enumerate(results, 1):
    print(f"\n{'='*70}")
    print(f"Q{i}: {r['query']}")
    print(f"\n[Retrieved top-3]")
    for _, row in r['retrieved'].head(3).iterrows():
        print(f"  - {row['name']:<35s} ({row['category']:<10s} £{row['price_gbp']:>3d})")
    print(f"\nA: {r['answer']}")
    print(f"[{r['elapsed']:.1f}s]")


Q1: I need a warm winter coat for under £200

[Retrieved top-3]
  - Highland Trench Coat                (coat       £195)
  - Frost Linen Shirt                   (shirt      £ 55)
  - Ember Quilted Jacket                (coat       £110)

A: Product 1: Highland Trench Coat (coat, £195) - This classic cotton trench is perfect for cold weather. It's breathable, durable, and comes with a belt to keep it secure.

Product 2: Frost Linen Shirt (shirt, £55) - A great choice if you're looking for something more casual but still comfortable during colder months. The lightweight fabric keeps your body cool without feeling too bulky.

Both of these options meet your criteria, so I'd recommend them both!
[15.9s]

Q2: Something to wear to a summer beach holiday

[Retrieved top-3]
  - Cassia Maxi Gown                    (dress      £ 65)
  - Frost Linen Shirt                   (shirt      £ 55)
  - Driftwood Straw Hat                 (accessory  £ 38)

A: I'd recommend the P0014 Frost Linen Shirt a

### B.1 · Grade each answer

For each of the 5 answers, rate on three criteria (write `1` = good, `0` = problem):

- **Grounded** — does the answer reference products that are in the retrieved set?
- **Relevant** — does it match what the customer asked for?
- **Honest** — if no good product exists, does it say so (vs. forcing a recommendation)?

In [11]:
# Fill in your grades manually based on the printed answers above.
# Replace each None with 1 (good), 0 (problem), or 0.5 (partial credit).
# The cell will refuse to score until you've graded every row.

grades = [
    # query, grounded, relevant, honest
    (BENCHMARK[0], None, None, None),
    (BENCHMARK[1], None, None, None),
    (BENCHMARK[2], None, None, None),
    (BENCHMARK[3], None, None, None),
    (BENCHMARK[4], None, None, None),
]

# Validate that you've filled them in
if any(v is None for _, *vs in grades for v in vs):
    print("⚠️  Some grades are still None. Fill them in based on the answers printed above.")
    print("    1 = good, 0 = problem, 0.5 = partial credit.")
    print("    Then re-run this cell.")
else:
    print(f"{'Query':45s} {'Ground':>7s} {'Relev':>7s} {'Honest':>7s}")
    print('-' * 75)
    totals = [0, 0, 0]
    for q, g, r, h in grades:
        print(f"  {q[:43]:45s} {g:>7} {r:>7} {h:>7}")
        totals[0] += g; totals[1] += r; totals[2] += h

    n = len(grades)
    print(f"\n{'TOTAL (out of '+str(n)+')':45s} {totals[0]:>7} {totals[1]:>7} {totals[2]:>7}")
    overall = sum(totals) / (3 * n)
    print(f"\nOverall quality score: {overall:.2f} / 1.00")
    print(f"\nTarget ≥ 0.7 for a shippable demo? {'✅ PASS' if overall >= 0.7 else '❌ Investigate why'}")

⚠️  Some grades are still None. Fill them in based on the answers printed above.
    1 = good, 0 = problem, 0.5 = partial credit.
    Then re-run this cell.


---

## Part C — Identify two improvement areas

Looking at your grading, find two specific weaknesses of your RAG system. For each:

1. **Describe** the weakness with a specific example from your benchmark
2. **Propose** a concrete fix (one to two sentences — don't just say "use a bigger model"; what specifically?)

### Weakness 1

**Description:** *(your answer)*

**Fix:** *(your answer)*

### Weakness 2

**Description:** *(your answer)*

**Fix:** *(your answer)*

### Optional reflection

Of the techniques in NB 04 Extensions (hallucination check, LLM re-rank), which would help your benchmark queries most? Why?

*(your answer)*
<details>
<summary>💡 Not sure what counts as a "weakness"? Click for common categories</summary>

Look for these patterns in your Part B results (each pairs with a concrete fix):

- **Retrieval miss** — the right product exists but wasn't in the top-5 (fix: better document text, larger K + re-rank, hybrid keyword+embedding search from L09)
- **Constraint ignored** — the answer names a product outside the retrieved list (fix: stricter grounding instruction, post-generation hallucination check from NB 04 E1)
- **Price/attribute blindness** — "under £200" ignored because embeddings don't do arithmetic (fix: a metadata filter on price BEFORE or AFTER retrieval)
- **Forced recommendation** — nothing fits but the model recommends anyway (fix: explicit "if nothing fits, say so" rule + a benchmark query designed to have no answer)
</details>

---

## Submission checklist

- [ ] Part A — `RAGSystem` implemented with retrieve + ask methods
- [ ] Part B — 5 benchmark queries run, answers printed
- [ ] Part B — Manual grading filled in (3 criteria × 5 queries)
- [ ] Part B target — Overall quality ≥ 0.7
- [ ] Part C — Two weaknesses identified with concrete fixes
- [ ] Part C — Optional reflection answered

Save the notebook with outputs and submit.